In [2]:
print("hello")

hello


In [3]:
import argparse, json, os, time, urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO

In [4]:
import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [5]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
COCO_DIR    = Path("./coco")
VAL_IMG_DIR = COCO_DIR / "val2017"          # ✅ removed "images/"
ANN_FILE    = COCO_DIR / "annotations/instances_val2017.json"
IMG_SIZE    = 640
DEVICE      = "cuda"      # "cuda" | "cpu" | "mps"
CONF        = 0.001       # low → recall-friendly (standard for mAP eval)
IOU         = 0.65
WARMUP      = 3
RESULTS_CSV = "results_yolov8.csv"

MODELS = [
    "yolov8n.pt",
    "yolov8s.pt",
    "yolov8m.pt",
    "yolov8l.pt",
    "yolov8x.pt",
]

In [6]:
# ── COCO DOWNLOAD ─────────────────────────────────────────────────────────────
def download_coco():
    COCO_DIR.mkdir(parents=True, exist_ok=True)
    files = {
        "val2017.zip": "http://images.cocodataset.org/zips/val2017.zip",
        "annotations_trainval2017.zip":
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    }
    for fname, url in files.items():
        dest = COCO_DIR / fname
        if not dest.exists():
            print(f"Downloading {fname} ...")
            urllib.request.urlretrieve(url, dest)
            with zipfile.ZipFile(dest) as z:
                z.extractall(COCO_DIR)
            print(f"  ✔ Extracted {fname}")

In [7]:
# ── COCO CATEGORY MAP ─────────────────────────────────────────────────────────
def build_id_map(ann_file):
    """Map 0-indexed YOLO class → COCO category_id."""
    with open(ann_file) as f:
        data = json.load(f)
    cats = sorted(data["categories"], key=lambda c: c["id"])
    return {i: c["id"] for i, c in enumerate(cats)}

In [8]:
# ── EVALUATION ────────────────────────────────────────────────────────────────
def coco_eval(coco_gt, predictions):
    if not predictions:
        return 0.0, 0.0
    coco_dt = coco_gt.loadRes(predictions)
    ev = COCOeval(coco_gt, coco_dt, "bbox")
    ev.evaluate()
    ev.accumulate()
    ev.summarize()
    # stats[0] = mAP@0.5:0.95 | stats[1] = mAP@0.5
    return round(ev.stats[1], 4), round(ev.stats[0], 4)

In [ ]:
# ── SINGLE MODEL BENCHMARK ────────────────────────────────────────────────────
def benchmark(model_name, img_ids, coco_gt, id_map):
    print(f"\n{'='*55}")
    print(f"  Model : {model_name}")
    print(f"{'='*55}")

    model = YOLO(model_name)

    # Model metadata
    weight_path = Path(model.ckpt_path) if hasattr(model, "ckpt_path") else Path(model_name)
    size_mb = weight_path.stat().st_size / 1e6 if weight_path.exists() else 0.0
    params_m = sum(p.numel() for p in model.model.parameters()) / 1e6

    # Warm-up
    dummy = str(next(VAL_IMG_DIR.glob("*.jpg")))
    for _ in range(WARMUP):
        model(dummy, imgsz=IMG_SIZE, device=DEVICE,
              conf=CONF, iou=IOU, verbose=False)

    predictions, latencies = [], []

    for img_id in tqdm(img_ids, desc=model_name, unit="img"):
        img_path = VAL_IMG_DIR / f"{img_id:012d}.jpg"
        if not img_path.exists():
            continue

        t0 = time.perf_counter()
        results = model(str(img_path), imgsz=IMG_SIZE, device=DEVICE,
                        conf=CONF, iou=IOU, verbose=False)[0]
        latencies.append((time.perf_counter() - t0) * 1000)

        boxes  = results.boxes.xyxy.cpu().numpy()
        scores = results.boxes.conf.cpu().numpy()
        cls    = results.boxes.cls.cpu().numpy().astype(int)

        for box, score, c in zip(boxes, scores, cls):
            x1, y1, x2, y2 = box
            predictions.append({
                "image_id":    img_id,
                "category_id": id_map.get(c, c + 1),
                "bbox":        [float(x1), float(y1),
                                float(x2 - x1), float(y2 - y1)],
                "score":       float(score),
            })

    map50, map5095 = coco_eval(coco_gt, predictions)
    avg_ms = round(float(np.mean(latencies)), 2)

    row = {
        "model":         model_name.replace(".pt", ""),
        "mAP@0.5":       map50,
        "mAP@0.5:0.95":  map5095,
        "speed_ms/img":  avg_ms,
        "size_MB":       round(size_mb, 1),
        "params_M":      round(params_m, 1),
    }
    print(f"\n  ▶ mAP@0.5        : {map50}")
    print(f"  ▶ mAP@0.5:0.95   : {map5095}")
    print(f"  ▶ Speed (ms/img)  : {avg_ms}")
    print(f"  ▶ Size (MB)       : {row['size_MB']}")
    print(f"  ▶ Params (M)      : {row['params_M']}")

    # ← ADDED: release model from GPU before returning
    del model
    return row

In [10]:
# ── MAIN ──────────────────────────────────────────────────────────────────────
import traceback
import gc
import torch

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--quick", action="store_true",
                        help="Run on 500 images instead of full 5k")
    args, _ = parser.parse_known_args()

    download_coco()

    coco_gt = COCO(str(ANN_FILE))
    img_ids = coco_gt.getImgIds()
    if args.quick:
        img_ids = img_ids[:500]
        print(f"[Quick mode] Using {len(img_ids)} images.")

    id_map = build_id_map(ANN_FILE)
    rows   = []

    for model_name in MODELS:
        try:
            row = benchmark(model_name, img_ids, coco_gt, id_map)
            rows.append(row)
        except Exception as e:
            print(f"  ⚠ Skipped {model_name}: {type(e).__name__}: {e}")
            traceback.print_exc()
        finally:                               # ← ADDED: runs after every model
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            print(f"  🧹 VRAM freed — "
                  f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | "
                  f"Cached: {torch.cuda.memory_reserved()/1e9:.2f} GB")

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)

    print(f"\n{'='*55}")
    print("  YOLOv8 — Final Results")
    print(f"{'='*55}")
    print(df.to_string(index=False))
    print(f"\n✅ Saved → {RESULTS_CSV}")

if __name__ == "__main__":
    main()

loading annotations into memory...
Done (t=0.51s)
creating index...
index created!

  Model : yolov8n.pt


yolov8n.pt: 100%|██████████| 5000/5000 [01:01<00:00, 81.74img/s]


Loading and preparing results...
DONE (t=0.77s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=23.60s).
Accumulating evaluation results...
DONE (t=4.25s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.367
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.521
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.396
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.178
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.406
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.519
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.305
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.498
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.544
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

yolov8s.pt: 100%|██████████| 5000/5000 [01:24<00:00, 59.15img/s]


Loading and preparing results...
DONE (t=0.71s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=21.55s).
Accumulating evaluation results...
DONE (t=3.72s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.442
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.608
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.475
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.254
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.490
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.601
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.344
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.559
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.605
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

...

Loading and preparing results...
DONE (t=0.63s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=20.23s).
Accumulating evaluation results...
DONE (t=3.28s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.493
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.662
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.534
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.310
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.546
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.653
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.370
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.604
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.647
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

yolov8l.pt: 100%|██████████| 5000/5000 [04:07<00:00, 20.24img/s]


Loading and preparing results...
DONE (t=0.58s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=17.76s).
Accumulating evaluation results...
DONE (t=2.83s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.519
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.687
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.563
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.346
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.572
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.687
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.384
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.624
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.668
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

yolov8x.pt: 100%|██████████| 5000/5000 [07:01<00:00, 11.87img/s]


Loading and preparing results...
DONE (t=0.60s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=18.23s).
Accumulating evaluation results...
DONE (t=3.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.530
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.698
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.575
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.349
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.584
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.694
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.389
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.633
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.675
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

In [21]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(RESULTS_CSV)

# Style the table
styled = (
    df.style
    .set_caption("YOLOv8 — Benchmark Results on COCO val2017")
    .format({
        "mAP@0.5":      "{:.4f}",
        "mAP@0.5:0.95": "{:.4f}",
        "speed_ms/img": "{:.1f} ms",
        "size_MB":      "{:.1f} MB",
        "params_M":     "{:.1f} M",
    })
    .highlight_max(subset=["mAP@0.5", "mAP@0.5:0.95"], color="black")
    .highlight_min(subset=["speed_ms/img", "size_MB", "params_M"], color="black")
    .set_properties(**{"text-align": "center", "font-size": "13px"})
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "15px"), ("font-weight", "bold"), ("padding", "10px")]},
        {"selector": "th",
         "props": [("background-color", "#000000"), ("color", "white"),
                   ("font-size", "13px"), ("text-align", "center"), ("padding", "8px")]},
        {"selector": "tr:hover",
         "props": [("background-color", "#0B0B0B")]},
    ])
    .hide(axis="index")
)

display(styled)

model,mAP@0.5,mAP@0.5:0.95,speed_ms/img,size_MB,params_M
yolov8n,0.5209,0.3668,11.5 ms,6.5 MB,3.2 M
yolov8s,0.6078,0.4416,16.2 ms,22.6 MB,11.2 M
yolov8m,0.6615,0.4929,33.2 ms,52.1 MB,25.9 M
yolov8l,0.6872,0.5188,48.6 ms,87.8 MB,43.7 M
yolov8x,0.6982,0.5295,83.4 ms,136.9 MB,68.2 M


In [11]:
print(torch.cuda.get_device_name(0))
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Cached:    {torch.cuda.memory_reserved() / 1e9:.2f} GB")

NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM: 8.6 GB
Allocated: 0.00 GB
Cached:    0.00 GB


In [3]:
from ultralytics import YOLO
import pandas as pd
from IPython.display import display

# ── Vos 20 classes cibles (Table 1) ───────────────────────────────
TARGET_CLASSES = [
    "bench", "bicycle", "billboard", "bookcase", "cabinetry",
    "car", "chair", "dog", "door", "fire hydrant",
    "furniture", "kitchen appliance", "person", "plant", "stairs",
    "stop sign", "table", "traffic light", "tree", "waste container"
]

model = YOLO("yolov8m.pt")

results = model.val(
    data="coco.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0,
)

# ── Construire la table ────────────────────────────────────────────
rows = []
names = results.names

for idx, ap in zip(results.ap_class_index, results.maps):
    class_name = names[idx].lower()
    if class_name in TARGET_CLASSES:
        rows.append({
            "Class":    names[idx].capitalize(),
            "Accuracy": round(float(ap) * 100, 1),
        })

df = pd.DataFrame(rows).sort_values("Class").reset_index(drop=True)

# Ligne Average
average_row = pd.DataFrame([{
    "Class":    "Average",
    "Accuracy": round(df["Accuracy"].mean(), 1),
}])

df_final = pd.concat([df, average_row], ignore_index=True)

# ── Afficher comme Table 1 dans Jupyter ───────────────────────────
styled = (
    df_final.style
    .set_caption("Table 1 — Accuracy of obstacle detection")
    .set_properties(**{"text-align": "left"}, subset=["Class"])
    .set_properties(**{"text-align": "right"}, subset=["Accuracy"])
    .apply(lambda x: ["font-style: italic; border-top: 1px solid black"] * len(x)
           if x.name == len(df_final) - 1 else [""] * len(x), axis=1)
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "13px"), ("text-align", "left")]},
        {"selector": "thead th", "props": [("border-top", "2px solid black"), ("border-bottom", "1px solid black")]},
        {"selector": "tbody tr:last-child td", "props": [("border-top", "1px solid black")]},
        {"selector": "tbody tr:last-child", "props": [("font-style", "italic")]},
        {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "350px")]},
        {"selector": "td, th", "props": [("padding", "4px 12px"), ("border", "none")]},
    ])
    .hide(axis="index")
)

display(styled)

Ultralytics 8.4.30  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLOv8m summary (fused): 92 layers, 25,886,080 parameters, 0 gradients, 78.9 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 733.8321.4 MB/s, size: 161.6 KB)
val: Scanning C:\Users\admin\PFE\PFE-Obstacle-Detection\coco\labels\val2017.cache... 5000 images, 48 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 1.8it/s 2:54<0.2ss
                   all       5000      36335      0.716       0.61      0.668      0.502
                person       2693      10777      0.821      0.745      0.829      0.618
               bicycle        149        314      0.742      0.525      0.628      0.403
                   car        535       1918      0.766      0.638      0.713        0.5
            motorcycle        159        367      0.805      0.677      0.792   

Class,Accuracy
Bench,32.900000
Bicycle,40.300000
Car,50.000000
Chair,39.000000
Dog,73.900000
Fire hydrant,75.200000
Person,61.800000
Stop sign,71.500000
Traffic light,32.200000
Average,53.000000
